# D2.4 · Containment at machine speed

**Function D — The Agentic SOC → The Agentic SOC — Response**  ·  *Security of AI*

Builds on **[D2.3 · Scoping an agentic incident](https://spbreed.github.io/cyber-commons/lessons/D2.3.html)**.

| | |
|---|---|
| Tools used | agentgateway, Keycloak |

## What this lesson is

**What it covers.** Exercise the ladder against a live misbehaving agent.

**Why a security engineer needs it.** Mass revocation takes down the business. The control it builds is: throttle → scope-reduce → reroute → force HITL → revoke → hard stop, in order.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

You have to stop it faster than it acts. That means the containment path — revoke, cut the gateway, kill the loop — is a thing built in advance, because improvising it takes longer than the incident does.

> **At CyberTravels.** You have to stop CyberTravels faster than it issues refunds. The containment path is something CyberTravels builds in advance, because improvising it takes longer than the incident.

## 2 · The framework

```
   containment paths, in order of how fast they actually work

   1  revoke the credential      seconds, if it is short-lived
   2  cut it at the gateway      seconds, if there is a gateway
   3  kill the loop              minutes, if you know where it runs
   4  disable the integration    hours

   built in advance. improvised, path 1 takes longer than the incident.
```

Containment has always been a race. With an agent, the other runner got much
faster and you did not.

The numbers decide the design. An agent operating at 300 actions per minute
completes 2,400 further actions during an eight-minute approval cycle, against
about 60 under automated containment. That ratio is the argument for
pre-authorised, automated revocation of non-human identities.

The asymmetry that makes it safe: revoking a **human's** access needs care,
because a false positive locks a person out mid-shift. Revoking a **non-human**
identity is cheap to get wrong — the agent re-requests, or an on-call re-enables
it in a minute. So the two should have different policies, and almost nowhere do.

## 3 · The procedure, as a skill

Eight minutes of approval is 2,400 actions. The skill races the rate against the delay, times the whole containment path — of which the revocation itself is twelve seconds — and classifies which signals may auto-revoke against a non-human subject.

In [ ]:
# skills/response/machine-speed-containment/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: machine-speed-containment
description: >-
  Race an agent's action rate against the approval delay in front of
  containment, time the whole containment path end to end, and decide which
  signals may auto-revoke. Use when containment requires a human and the subject
  acts hundreds of times a minute.
allowed-tools: Read, Grep, Glob
---

# Eight minutes of approval is 2,400 actions

Containment that waits for a person is measured against a subject that does not
wait. The arithmetic is not close, and it is the argument for pre-authorising
revocation for a named set of signals — with the false-revocation cost stated,
because that is the objection.

## When to use this

Designing containment for agent workloads, and after any incident where the
containment step was correct and late.

## Procedure

**1 — Measure the subject's rate.** Actions per minute, observed rather than
designed. Multiply by the approval delay to get the actions taken while
somebody decides.

**2 — Time the whole path, not the revocation.** Detection, triage, decision,
approval, execution, propagation. The revocation itself is usually seconds and
the path is usually minutes; reporting only the last step makes the problem
invisible.

**3 — Find the dominant term.** It is almost always human approval or
propagation delay, and it is almost never the API call. Optimise the dominant
term or nothing changes.

**4 — Classify signals for auto-revocation.** For each, its precision and what a
false revocation costs. High-precision signals against a non-human subject are
the candidates: revoking an agent's token wrongly costs a restarted run.

**5 — Set the policy asymmetrically.** Auto-revoke agent credentials on
high-precision signals; keep a human in front of anything that affects a person's
access. State both halves so the policy survives review.

## Output contract

```json
{
  "race": {"actions_per_min": 0, "approval_minutes": 0, "actions_during_approval": 0},
  "path": [{"step": "str", "seconds": 0}],
  "total_seconds": 0,
  "dominant_step": "str",
  "signals": [{"name": "str", "precision": 0.0, "subject": "agent|human",
               "auto_revoke": false, "false_revocation_cost": "str"}]
}
```

## Failure modes

- **Timing the revocation.** It is the fast part.
- **One policy for agents and people.** The costs differ by orders of
  magnitude.
- **Auto-revoking on a low-precision signal.** Precision is the entry
  requirement.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/response/machine-speed-containment/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/response/machine-speed-containment/scripts/machine_speed_containment.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Race an agent's action rate against an approval delay, time the whole containment path, and decide what may auto-revoke.

This is the executable half of the `machine-speed-containment` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

def race(actions_per_min, human_minutes, auto_seconds=12):
    manual = actions_per_min * human_minutes
    auto   = actions_per_min * (auto_seconds/60)
    return {"manual": round(manual), "auto": round(auto),
            "ratio": round(manual/max(auto, 1e-9), 1)}

print(f"{'agent rate':>13}{'human 8min':>13}{'auto 12s':>11}{'ratio':>8}")
print("-" * 46)
for rate in (30, 120, 300, 1200):
    r = race(rate, 8)
    print(f"{rate:>9}/min{r['manual']:>13}{r['auto']:>11}{r['ratio']:>8}×")
print("\nAt 300/min an 8-minute approval costs 2,400 further actions.")

PATH = [
 ("detection fires",              8,   "rule evaluation + SIEM ingestion lag"),
 ("analyst picks it up",          240, "queue depth at 02:00"),
 ("analyst decides to contain",   180, "confirming it is not a false positive"),
 ("approval requested",           480, "on-call manager, out of hours"),
 ("revocation executed",          12,  "the only step anyone measures"),
]
total = sum(s for _, s, _ in PATH)
print(f"{'step':30s}{'seconds':>9}  why")
print("-" * 74)
for name, secs, why in PATH:
    print(f"{name:30s}{secs:>9}  {why}")
print(f"{'TOTAL':30s}{total:>9}  = {total/60:.0f} minutes")
print(f"\nat 300 actions/min that is {300*total/60:,.0f} further actions.")
print("The 12-second revocation is 1.3% of the elapsed time. Optimising it")
print("is not where the win is.")

SIGNALS = {
 "reached the cloud metadata service": 0.99,
 "read a path matching */.ssh/* or */.aws/*": 0.97,
 "egress to a host not on the allowlist": 0.90,
 "tool-call rate 20× its own baseline": 0.75,
 "activity outside usual hours": 0.30,
}
THRESHOLD = 0.70

def policy(signal, subject_is_human):
    conf = SIGNALS[signal]
    if subject_is_human:
        return f"page on-call (confidence {conf:.2f}) — human lockout needs a person"
    if conf >= THRESHOLD:
        return f"AUTO-REVOKE (confidence {conf:.2f}) — no approval in the path"
    return f"alert only (confidence {conf:.2f} < {THRESHOLD})"

for s in SIGNALS:
    print(f"{s:44s}{policy(s, False)}")
print()
print(f"{'same signal, human subject':44s}"
      f"{policy('reached the cloud metadata service', True)}")

auto_path = [("detection fires", 8), ("policy evaluates", 1), ("revocation executed", 12)]
auto_total = sum(s for _, s in auto_path)
print(f"\nautomated path: {auto_total}s vs manual {total}s "
      f"({total/auto_total:.0f}× faster)")
print(f"actions prevented at 300/min: {300*(total-auto_total)/60:,.0f}")
assert auto_total < total / 10

# Verify: model the cost of getting it wrong, which is what makes it safe.
def cost_of_false_revocation(subject_is_human, agent_can_rerequest=True):
    if subject_is_human:
        return {"impact": "person locked out mid-shift", "recovery": "helpdesk, 20-60 min",
                "cost": "high"}
    if agent_can_rerequest:
        return {"impact": "task fails, agent re-requests with a reason (A2.4)",
                "recovery": "seconds to minutes", "cost": "low"}
    return {"impact": "agent stops until an on-call re-enables it",
            "recovery": "minutes", "cost": "moderate"}

for label, human in (("human subject", True), ("non-human identity", False)):
    c = cost_of_false_revocation(human)
    print(f"{label:22s}{c['cost']:10s}{c['impact']}")
print("\nThat asymmetry is the entire justification for two different policies.")

## What you just proved

The race table shows 2,400 versus 60 actions at 300/min for an eight-minute approval. The full containment path totals about 920 seconds, of which the revocation itself is 12. Four of five signals auto-revoke for non-human identities and none do for a human subject, cutting the path to 21 seconds and preventing roughly 4,500 actions.

## Your turn

Time your own containment path end to end, step by step. The revocation is almost never the slow part — queue depth and approval are, and both are policy choices rather than technical limits.

---

**Next → [D2.5 · Replay and forensics](https://spbreed.github.io/cyber-commons/lessons/D2.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/D2.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/D2.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*